In [ ]:
import torch
import torchvision.models as models

# 打印 MobileNet V2 的网络结构
print("=" * 60)
print("MobileNet V2 网络结构")
print("=" * 60)
model = models.mobilenet_v2(weights=None)
print(model)


In [ ]:
import torch
import torchvision.models as models
import torch.nn.utils.prune as prune

model = models.mobilenet_v2(weights=None)  # 重新实例化，确保是未剪枝的干净模型

# ============================================================
# Part 1：讲义版 —— 单层局部剪枝 + 机制演示
# ============================================================
module = model.features[0][0]  # 第一层卷积
fc = model.classifier[1]       # 分类头全连接层

# 1) 随机非结构化剪枝：随机选 30% 的权重置 0（不推荐，有偶然性）
prune.random_unstructured(module, name="weight", amount=0.3)

# 2) L1 非结构化剪枝：剪掉绝对值（L1范数）最小的权重（推荐）
prune.l1_unstructured(fc, name="bias", amount=3)  # amount=3 表示剪 3 个

print("=" * 60)
print("Part 1：剪枝机制演示")
print("=" * 60)
# 剪枝后：原始权重改名为 weight_orig，保存在 named_parameters 中
print("module 参数(出现 weight_orig):", [n for n, _ in module.named_parameters()])
# 剪枝掩码 weight_mask 保存在 named_buffers 中
print("module 缓冲区(出现 weight_mask):", [n for n, _ in module.named_buffers()])
print("fc 参数(出现 bias_orig):", [n for n, _ in fc.named_parameters()])
print("fc 缓冲区(出现 bias_mask):", [n for n, _ in fc.named_buffers()])
print(f"module 第一层卷积稀疏度: {(module.weight_mask == 0).sum().item() / module.weight_mask.numel() * 100:.2f}%")
print(f"fc 被剪掉的 bias 个数: {(fc.bias_mask == 0).sum().item()}")

# 固化剪枝结果：把 mask 乘进 weight，删除 _orig 和 _mask
prune.remove(module, 'weight')
prune.remove(fc, 'bias')
print("prune.remove 后 module 参数(无 weight_orig):", [n for n, _ in module.named_parameters()])
print(f"固化后 module.weight 中 0 的个数: {(module.weight == 0).sum().item()}")

# ============================================================
# Part 2：全局 L1 剪枝 + 效果统计
# ============================================================
def count_parameters(model):
    """统计模型总参数量"""
    return sum(p.numel() for p in model.parameters())

conv_layers = [(m, 'weight') for m in model.modules() if isinstance(m, torch.nn.Conv2d)]

print("\n" + "=" * 60)
print(f"Part 2：全局 L1 剪枝（共 {len(conv_layers)} 个卷积层）")
print("=" * 60)
print(f"剪枝前总参数量: {count_parameters(model):,}")

# 剪掉所有权重中 L1 范数最小的 30%（跨层统一排序）
prune.global_unstructured(
    parameters=conv_layers,
    pruning_method=prune.L1Unstructured,
    amount=0.3,  # 剪枝比例 30%
)

total_params = 0
total_zero = 0
for m, _ in conv_layers:
    mask = m.weight_mask
    total_params += mask.numel()
    total_zero += (mask == 0).sum().item()

print(f"被置零的权重数: {total_zero:,}")
print(f"整体稀疏度: {total_zero / total_params * 100:.2f}%")

print("\n前 8 个卷积层的稀疏度:")
for i, (m, _) in enumerate(conv_layers[:8]):
    mask = m.weight_mask
    print(f"  卷积层 {i}: {(mask == 0).sum().item() / mask.numel() * 100:6.2f}%")

# ============================================================
# Part 3：结构化剪枝 —— 整条通道地剪
# ============================================================
module = model.features[0][0]  # 第一层卷积
print("\n" + "=" * 60)
print("Part 3：结构化剪枝（按通道剪）")
print("=" * 60)
print(f"剪枝前 weight 形状: {module.weight.shape}")

# 结构化剪枝：整行/整列/整个通道地剪，多一个 dim 参数
# n=1 表示用 L1 范数评估，dim=0 表示按"输出通道"维度整条剪
prune.ln_structured(module, name="weight", amount=0.3, n=1, dim=0)
prune.remove(module, 'weight')

# 统计被整条剪掉的通道数：该通道所有值求和为 0 说明整条被剪掉
pruned_channels = (module.weight.sum(dim=(1, 2, 3)) == 0).sum().item()
print(f"被整条剪掉的通道数: {pruned_channels}")
print(f"剪枝后 weight 形状: {module.weight.shape}（形状不变，只是整条通道变 0）")
print(module.weight.shape)

# ============================================================
# Part 4：循环剪枝整个网络 —— 迭代式剪枝
# ============================================================
# 与"一次性剪掉 50%"不同：循环剪枝每轮只剪一点、累计多轮，
# 避免一刀切误伤重要权重，精度损失更小（工业界常用做法）
model2 = models.mobilenet_v2(weights=None)  # 重新实例化干净模型

print("\n" + "=" * 60)
print("Part 4：循环剪枝整个网络（10 轮，累计剪到 50%）")
print("=" * 60)

conv_layers2 = [(m, 'weight') for m in model2.modules() if isinstance(m, torch.nn.Conv2d)]

def get_sparsity(layers):
    """统计所有卷积层的整体稀疏度"""
    total = sum(m.weight.numel() for m, _ in layers)
    zeros = sum((m.weight == 0).sum().item() for m, _ in layers)
    return zeros / total * 100

print(f"剪枝前稀疏度: {get_sparsity(conv_layers2):.2f}%")
n_rounds = 10     # 总轮数
target = 0.5      # 累计剪到 50%
for r in range(1, n_rounds + 1):
    target_r = target * r / n_rounds  # 本轮累计目标：5%, 10%, ..., 50%
    current = get_sparsity(conv_layers2) / 100
    # torch 循环剪枝时，amount 是"剪掉当前剩余权重的比例"（会复合增长），
    # 为精确达到累计目标，用公式换算：amount = (目标 - 当前) / (1 - 当前)
    amount = (target_r - current) / (1 - current)
    prune.global_unstructured(
        parameters=conv_layers2,
        pruning_method=prune.L1Unstructured,
        amount=amount,
    )
    print(f"  第 {r:2d} 轮（累计 {target_r * 100:3.0f}%）：整体稀疏度 {get_sparsity(conv_layers2):5.2f}%")

# 固化所有层的剪枝结果
for m, _ in conv_layers2:
    prune.remove(m, 'weight')
print(f"\n循环剪枝完成：最终稀疏度 {get_sparsity(conv_layers2):.2f}%")
print(f"最终总参数量: {count_parameters(model2):,}（结构不变，只是大量权重为 0）")

# 用人话理解

**一句话总结：** 这个单元格在演示"怎么给神经网络做剪枝"——把模型里**不重要的权重偷偷清零**，让模型更省内存、跑得更快，而精度基本不掉。

## 先打个比方 🍎

把模型想象成一个大书架，上面有几百万本书（= 权重参数）。书架太满，手机装不下。剪枝就是：**找出那些"从没被人翻开过"的书（数值接近 0 的权重），把它们标记为废纸（清零）**，书架看起来还是那样，但真正有用的书反而更突出、更好找。

---

## 四个 Part 分别干了啥？

### 🧩 Part 1：先挑两个"代表"练练手（局部剪枝）
- 抓了**第一个卷积层**（`module = model.features[0][0]`）和**分类头**（`fc = model.classifier[1]`）两个代表
- 用了两种剪法：
  - **随机剪**（`random_unstructured`）：闭着眼睛随机挑 30% 清零 —— 不推荐，因为可能误伤重要权重
  - **L1 剪**（`l1_unstructured`）：按"绝对值最小 = 最不重要"挑 3 个 bias 清零 —— 推荐
- 重点看**剪完内部发生了什么**：
  - 原来的权重改名成 `weight_orig`（原封不动存着）
  - 多了一张 `weight_mask` 掩码表（1 = 保留，0 = 清零）
  - 真正用的权重 = 原权重 × 掩码表
- 最后 `prune.remove`：把掩码"焊死"进权重，删掉临时表

> **大白话**：Part 1 是"先拿两个层做实验，看清楚剪枝的记账方式"。

### 🌍 Part 2：给整个网络"统一体检"（全局剪枝）
- 找出**全部 52 个卷积层**，把 350 万个权重**放到一起排名**，谁的绝对值最小就先剪谁
- 一次剪掉全局最小的 30%，然后统计：
  - 整体稀疏度 = 30.00%（被清零的比例）
  - 每个层被剪得不一样多（有的 45%，有的才 4%）—— 说明"不重要的层多剪、重要的层少剪"

> **大白话**：Part 2 是"全公司统一裁员，按能力（权重绝对值）排名，能力差的先走"。

### 🧱 Part 3：整条通道一起剪（结构化剪枝）
- 前面剪的是**单个权重**（零散置 0），这次是**整条通道**（`dim=0`）：把某个输出通道的 27 个权重**整组清零**
- 结果：第一层 32 条通道里，**10 条整组变 0**，但形状没变（通道还在，只是内容全空了）

> **大白话**：非结构化剪枝像"撒胡椒面"（到处零星清零）；结构化剪枝像"拆墙"（整面墙推倒），将来可以把整条通道真正删掉，省内存效果更明显。

### 🔁 Part 4：循环剪枝（迭代式）
- 不是一次剪 50%，而是**每轮只剪一点、累计 10 轮**剪到 50%
- 每轮输出稀疏度：5% → 10% → ... → 50%，逐步推进
- 这样做的意义：**每剪完一轮可以微调一下再继续**，比"一刀切"剪 50% 精度损失更小

> **大白话**：一次剪太多容易"伤筋动骨"，慢慢剪、边剪边恢复，才是工业界的做法。

---

## 一句话记住这 4 个 Part

| Part | 剪什么 | 剪的范围 | 关键词 |
|:---:|------|------|------|
| 1 | 单个权重 | 两个代表层 | `weight_orig` / `weight_mask` / `remove` |
| 2 | 单个权重 | 全部 52 层（全局） | 稀疏度统计 |
| 3 | 整条通道 | 单层 | `dim=0` 结构化 |
| 4 | 单个权重 | 全部层、分 10 轮 | 迭代 + 累计目标 |

In [ ]:
import os
import torch
from torch import nn
import torch.nn.utils.prune as prune
import torchvision
from torchvision import models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- 网络：MobileNetV2 ----------
def build_mobilenet():
    model = models.mobilenet_v2(weights=None)  # 不用 ImageNet 预训练权重：输入和类别都对不上
    # 1：第一层卷积 3 通道（RGB）-> 1 通道（MNIST 灰度图）
    model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
    # 2：分类头 1000 类（ImageNet）-> 10 类（MNIST）
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 10)
    return model

# ---------- 准确率 + 平均推理时间 ----------
def test(model, name):
    test_dataset = torchvision.datasets.MNIST(root="./data/MNIST", train=False,
                                              transform=transforms.ToTensor(),
                                              download=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=1024, shuffle=True)
    model.eval()
    num_correct, num_samples, times = 0, 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            s = time.time()
            preds = model(x)
            times += time.time() - s
            num_correct += (preds.max(1).indices == y).sum()
            num_samples += preds.size(0)
    acc = (num_correct / num_samples).item()
    print("{} 准确率 acc:{:.6f} 平均推理时间:{:.6f}".format(name, acc, times / num_samples))

# ---------- baseline：从零训练 6 轮 ----------
def train_model(model, epochs=6):
    train_dataset = torchvision.datasets.MNIST(root="./data/MNIST", train=True,
                                               transform=transforms.ToTensor(),
                                               download=True)
    train_loader = DataLoader(dataset=train_dataset, batch_size=1024, shuffle=True)
    loss_fun = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    for epoch in range(epochs):
        model.train()
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            out = model(data)
            loss = loss_fun(out, target)
            opt.zero_grad()
            loss.backward()
            opt.step()
        print("epoch:{} 训练完成".format(epoch + 1))
    return model

# ---------- 剪枝：卷积层剪 50%，全连接层剪 30%，并固化 ----------
def prune_model(model):
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            prune.l1_unstructured(module, name='weight', amount=0.5)
            prune.remove(module, 'weight')  # 固化，删除 weight_orig 和 weight_mask
        elif isinstance(module, torch.nn.Linear):
            prune.l1_unstructured(module, name='weight', amount=0.3)
            prune.remove(module, 'weight')
    return model

# ---------- 微调 1 轮 ----------
def fine_tune(model, epochs=1):
    train_dataset = torchvision.datasets.MNIST(root="./data/MNIST", train=True,
                                               transform=transforms.ToTensor(),
                                               download=True)
    train_loader = DataLoader(dataset=train_dataset, batch_size=1024, shuffle=True)
    loss_fun = nn.CrossEntropyLoss()
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    for epoch in range(epochs):
        model.train()
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            out = model(data)
            loss = loss_fun(out, target)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model

if __name__ == '__main__':
    os.makedirs("params", exist_ok=True)
    model = build_mobilenet().to(device)  # 第一步：网络
    model = train_model(model)            # 第二步：训练 baseline
    test(model, "剪枝前")
    torch.save(model.state_dict(), "params/model_mobilenet.pt")

    model = prune_model(model)            # 第三步：剪枝
    test(model, "剪枝后")
    torch.save(model.state_dict(), "params/model_prune.pt")

    model = fine_tune(model)              # 第四步：微调
    test(model, "微调后")
    torch.save(model.state_dict(), "params/model_retrain.pt")

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# ============ 1. 定义 Teacher（大网络）和 Student（小网络）============
class Teacher(nn.Module):
    """大而强：3 层卷积 + 2 层全连接（约 240 万参数）"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),   # 16x16
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),  # 8x8
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),  # 4x4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

class Student(nn.Module):
    """小而巧：2 层卷积 + 1 层全连接（约 27 万参数）"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),  # 16x16
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),  # 8x8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def count_params(model):
    return sum(p.numel() for p in model.parameters())

teacher = Teacher().to(device)
student_scratch = Student().to(device)   # 对照组：自己练
student_distill = Student().to(device)   # 实验组：跟老师学
print(f"Teacher 参数量: {count_params(teacher):,}")
print(f"Student 参数量: {count_params(student_scratch):,}")
print(f"参数比: Teacher / Student ≈ {count_params(teacher) / count_params(student_scratch):.1f} 倍")

# ============ 2. 数据：本地 CIFAR-10 ============
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),  # CIFAR-10 均值/方差
])
train_set = torchvision.datasets.CIFAR10(root="D:\\Medicaldata", train=True,
                                         transform=transform, download=False)
test_set = torchvision.datasets.CIFAR10(root="D:\\Medicaldata", train=False,
                                        transform=transform, download=False)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=512, shuffle=False)
print(f"训练集: {len(train_set)} 张, 测试集: {len(test_set)} 张")
print(f"类别: {train_set.classes}")

# ============ 3. 工具函数 ============
def evaluate(model, name):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    acc = correct / total
    print(f"  {name:22s} 准确率: {acc:.4f}")
    return acc

def train_plain(model, epochs, name):
    """普通训练：只用真实标签（CrossEntropy）"""
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = F.cross_entropy(out, y)
            opt.zero_grad(); loss.backward(); opt.step()
        print(f"  [{name}] epoch {epoch}/{epochs} 完成")
    return model

def train_distill(student, teacher, epochs, T=6.0, alpha=0.7):
    """蒸馏训练：学生同时学 真实标签(硬标签) + 老师输出(软标签)"""
    opt = torch.optim.Adam(student.parameters(), lr=1e-3)
    teacher.eval()  # 老师冻结，只负责出答案
    for epoch in range(1, epochs + 1):
        student.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            stu_out = student(x)
            with torch.no_grad():
                tea_out = teacher(x)  # 老师的软答案（概率分布）
            # 软标签损失：KL 散度，用温度 T 把分布"软化"（放大类间相似信息）
            soft_loss = F.kl_div(
                F.log_softmax(stu_out / T, dim=1),
                F.softmax(tea_out / T, dim=1),
                reduction="batchmean",
            ) * (T * T)
            # 硬标签损失：跟真实答案对齐
            hard_loss = F.cross_entropy(stu_out, y)
            loss = alpha * soft_loss + (1 - alpha) * hard_loss
            opt.zero_grad(); loss.backward(); opt.step()
        print(f"  [学生-蒸馏] epoch {epoch}/{epochs} 完成")
    return student

# ============ 4. 开训 ============
print("\n① 训练 Teacher（大网络，训 10 轮）...")
train_plain(teacher, 10, "Teacher")

print("\n② 训练 Student-A：自己练，不跟老师学（训 10 轮）...")
train_plain(student_scratch, 10, "Student-自己练")

print("\n③ 训练 Student-B：跟着老师学（蒸馏，训 10 轮）...")
train_distill(student_distill, teacher, 10)

# ============ 5. 最终对比 ============
print("\n" + "=" * 56)
print("最终对比")
print("=" * 56)
acc_teacher = evaluate(teacher, "Teacher(大网络)")
acc_scratch = evaluate(student_scratch, "Student 自己练")
acc_distill = evaluate(student_distill, "Student 蒸馏")

print("\n结论：")
print(f"  蒸馏学生比自学学生高 {acc_distill - acc_scratch:+.4f}")
print(f"  蒸馏学生距离大网络差 {acc_teacher - acc_distill:.4f}，但参数只有 {count_params(student_distill) / count_params(teacher) * 100:.1f}%")

In [3]:
import os

# 把手动安装的 TensorRT 的 lib 目录加入 DLL 搜索路径（必须放在 import 之前）
os.environ["PATH"] = r"D:\TensorRT-10.10.0.31\TensorRT-10.10.0.31\lib" + os.pathsep + os.environ["PATH"]

from ultralytics import YOLO
import tensorrt as trt

print(f"TensorRT 版本: {trt.__version__}")

# ============================================================
# YOLOv8 模型量化导出（用 week12 里的官方预训练 yolov8n.pt）
# ============================================================
# 模型：yolov8n.pt（官方预训练，COCO 80 类，imgsz=640）
yolo_pt = r"D:\project\step1\week12\yolov8n.pt"
model = YOLO(yolo_pt)
print(f"任务类型: {model.task}")
print(f"类别数: {len(model.names)}")

# ============================================================
# Part 1：FP16 半精度 TensorRT engine
# ============================================================
print("\n① FP16:TensorRT engine（半精度，体积减半、精度几乎无损）...")
model.export(format="engine", half=True)
# 重命名，避免被后续 INT8 engine 覆盖
os.replace(r"D:\project\step1\week12\yolov8n.engine",
           r"D:\project\step1\week12\yolov8n_fp16.engine")
print("  → 已重命名为 yolov8n_fp16.engine")

# ============================================================
# Part 2：INT8 量化 TensorRT engine（重点）
# ============================================================
print("\n② INT8:TensorRT engine（int8 量化，data 指定校准数据集）...")
print("   官方建议校准数据 1000 张以上，精度损失最小（这里用 coco8 演示）")
model.export(format="engine", int8=True, data="coco8.yaml")
os.replace(r"D:\project\step1\week12\yolov8n.engine",
           r"D:\project\step1\week12\yolov8n_int8.engine")
print("  → 已重命名为 yolov8n_int8.engine")

# ============================================================
# Part 3：INT8 OpenVINO（Intel CPU / 核显）
# ============================================================
print("\n③ INT8:OpenVINO（Intel CPU / 核显）...")
model.export(format="openvino", int8=True, data="coco8.yaml")

print("\n" + "=" * 60)
print("导出完成！生成的文件在 D:\\project\\step1\\week12 下：")
print("  yolov8n_fp16.engine（TensorRT FP16）| yolov8n_int8.engine（TensorRT INT8）")
print("  yolov8n_int8_openvino_model/（OpenVINO INT8）")
print("=" * 60)

TensorRT 版本: 10.10.0.31
任务类型: detect
类别数: 80

① FP16:TensorRT engine（半精度，体积减半、精度几乎无损）...
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.113  Python-3.11.14 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)


YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'D:\project\step1\week12\yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)

ONNX: starting export with onnx 1.21.0 opset 17...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  1.1s, saved as 'D:\project\step1\week12\yolov8n.onnx' (12.3 MB)

TensorRT: starting export with TensorRT 10.10.0.31...
TensorRT: input "images" with shape(1, 3, 640, 640) DataType.FLOAT
TensorRT: output "output0" with shape(1, 84, 8400) DataType.FLOAT
TensorRT: building FP16 engine as D:\project\step1\week12\yolov8n.engine
TensorRT: export success  322.8s, saved as 'D:\project\step1\week12\yolov8n.engine' (9.6 MB)

Export complete (323.0s)
Results saved to D:\project\step1\week12\yolov8n.engine
Predict:         yolo predict task=detect model=D:\project\step1\week12\yolov8n.engine imgsz=640 quantize=16
Validate:        yolo val task=detect model=D:\projec

Output()

Output()

OpenVINO: export success  8.4s, saved as 'D:\project\step1\week12\yolov8n_int8_openvino_model\' (3.5 MB)

Export complete (8.7s)
Results saved to D:\project\step1\week12\yolov8n_int8_openvino_model
Predict:         yolo predict task=detect model=D:\project\step1\week12\yolov8n_int8_openvino_model\ imgsz=640 
Validate:        yolo val task=detect model=D:\project\step1\week12\yolov8n_int8_openvino_model\ imgsz=640 data=coco.yaml  
Visualize:       https://netron.app

导出完成！生成的文件在 D:\project\step1\week12 下：
  yolov8n_fp16.engine（TensorRT FP16）| yolov8n_int8.engine（TensorRT INT8）
  yolov8n_int8_openvino_model/（OpenVINO INT8）
